# Baseline CNN - Dominio Temporal

Carga formas de onda crudas y entrena una CNN1D. Sin MFCC ni espectrogramas.

## 1. Importacion de Librerias

In [ ]:
import json
import librosa
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
print(f'TensorFlow: {tf.__version__}')

## 2. Configuracion y Carga de Particiones

Selecciona dataset y fold. El JSON ya tiene los 5 folds precalculados con separacion subject-wise.

In [ ]:
DATASET  = 'neurovoz'  # 'neurovoz' o 'pc-gita'
NUM_FOLD = 0           # 0, 1, 2, 3 o 4

PROJECT_ROOT  = Path().resolve().parent.parent
SPLITS_PATH   = PROJECT_ROOT / 'data' / 'data_splits.json'
BASE_DATA_DIR = PROJECT_ROOT / 'data' / 'processed'

with open(SPLITS_PATH, 'r') as f:
    splits_dict = json.load(f)

fold_data = splits_dict[DATASET][NUM_FOLD]
print(f'Fold {NUM_FOLD} cargado para {DATASET.upper()}:')
print(f'  Train : {len(fold_data["train_files"])} audios | {len(fold_data["train_subjs"])} pacientes')
print(f'  Valid : {len(fold_data["val_files"])} audios | {len(fold_data["val_subjs"])} pacientes')
print(f'  Test  : {len(fold_data["test_files"])} audios | {len(fold_data["test_subjs"])} pacientes')

In [ ]:
# -- Distribucion de clases y vocales en TRAIN --
from pathlib import Path as _P
from collections import Counter as _C

def _get_label(f):
    return 'Sano'    if 'Control' in str(f) else 'Enfermo'

def _get_vowel(f):
    parts = _P(f).parts
    # estructura: dataset / clase / vocal / fichero.wav
    return parts[2] if len(parts) >= 4 else '?'

train_files = fold_data['train_files']

# Conteos
class_counts = _C(_get_label(f) for f in train_files)
vowel_counts  = _C(_get_vowel(f)  for f in train_files)

# Distribucion por clase Y vocal combinada
cv_counts = _C((_get_label(f), _get_vowel(f)) for f in train_files)

print(f'=== TRAIN  (Fold {NUM_FOLD} | {DATASET.upper()}) ===')
print(f'Total audios: {len(train_files)}')
print()
print('-- Clases --')
for cls, n in sorted(class_counts.items()):
    print(f'  {cls:10s}: {n:4d}  ({n/len(train_files)*100:.1f}%)')
print()
print('-- Vocales --')
for v, n in sorted(vowel_counts.items()):
    print(f'  {v:3s}: {n:4d}  ({n/len(train_files)*100:.1f}%)')
print()
print('-- Clase x Vocal --')
for (cls, v), n in sorted(cv_counts.items()):
    print(f'  {cls:10s} + {v}: {n:4d}')

## 3. Carga de Formas de Onda Crudas

Los archivos en `data/processed/` ya estan normalizados en amplitud y recortados por `preprocessing.py`. Aqui solo se carga la senal cruda 1D sin ninguna transformacion frecuencial.

In [ ]:
SR_TARGET   = 22050
MAX_SAMPLES = int(0.48891 * SR_TARGET)  # aprox 10780 muestras

def load_waveform(relative_path, base_dir):
    """
    Carga la forma de onda cruda de un .wav ya preprocesado.
    Solo padding/crop de seguridad. No se extrae ninguna caracteristica.
    """
    y, _ = librosa.load(base_dir / relative_path, sr=SR_TARGET)
    if len(y) > MAX_SAMPLES:
        y = y[:MAX_SAMPLES]
    elif len(y) < MAX_SAMPLES:
        y = np.pad(y, (0, MAX_SAMPLES - len(y)), mode='constant')
    return y

def get_label(relative_path):
    return 0 if 'Control' in str(relative_path) else 1

def build_dataset_tensors(file_list, base_dir):
    X, y = [], []
    for f in file_list:
        X.append(load_waveform(f, base_dir))
        y.append(get_label(f))
    X = np.array(X)  # (N, MAX_SAMPLES)
    y = np.array(y)  # (N,)
    return np.expand_dims(X, axis=-1), y  # Conv1D necesita (N, pasos, canales)

In [ ]:
print('Cargando Train...')
X_train, y_train = build_dataset_tensors(fold_data['train_files'], BASE_DATA_DIR)
print('Cargando Validacion...')
X_val, y_val = build_dataset_tensors(fold_data['val_files'], BASE_DATA_DIR)
print('Cargando Test...')
X_test, y_test = build_dataset_tensors(fold_data['test_files'], BASE_DATA_DIR)
print(f'\nTensores listos.')
print(f'  X_train : {X_train.shape}  |  y_train : {y_train.shape}')
print(f'  X_val   : {X_val.shape}  |  y_val   : {y_val.shape}')
print(f'  X_test  : {X_test.shape}  |  y_test  : {y_test.shape}')

## 4. Construccion de la CNN (Conv1D)

Tres bloques Conv1D-MaxPooling1D aprenden patrones directamente de la forma de onda cruda.

In [ ]:
def build_cnn_model(input_shape):
    model = models.Sequential([
        layers.Input(shape=input_shape),

        # Bloque 1: patrones basicos de la onda
        layers.Conv1D(32,  kernel_size=16, activation='relu', padding='same'),
        layers.MaxPooling1D(pool_size=4),
        layers.Dropout(0.2),

        # Bloque 2: patrones de nivel medio
        layers.Conv1D(64,  kernel_size=16, activation='relu', padding='same'),
        layers.MaxPooling1D(pool_size=4),
        layers.Dropout(0.25),

        # Bloque 3: patrones abstractos
        layers.Conv1D(128, kernel_size=16, activation='relu', padding='same'),
        layers.MaxPooling1D(pool_size=4),
        layers.Dropout(0.25),

        # Clasificador
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),

        # Salida: prob [0,1] -> Sano (0) o Patologico (1)
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

model = build_cnn_model(X_train.shape[1:])
model.summary()

## 5. Entrenamiento

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    epochs=40,
    batch_size=16,
    validation_data=(X_val, y_val),
    callbacks=[early_stop],
    verbose=1
)

## 6. Evaluacion sobre Validacion

> Todas las metricas se calculan sobre el **conjunto de validacion** (X_val / y_val).

In [ ]:
loss, acc = model.evaluate(X_val, y_val, verbose=0)
print(f'Accuracy en Validacion: {acc*100:.2f}%')
print(f'Loss en Validacion    : {loss:.4f}')

### 6.1 Curvas de Aprendizaje

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['accuracy'],     label='Train',      color='steelblue')
axes[0].plot(history.history['val_accuracy'], label='Validacion', color='darkorange')
axes[0].set_title('Precision (Accuracy)')
axes[0].set_xlabel('Epoca')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history.history['loss'],     label='Train',      color='steelblue')
axes[1].plot(history.history['val_loss'], label='Validacion', color='darkorange')
axes[1].set_title('Error (Loss)')
axes[1].set_xlabel('Epoca')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 6.2 Matriz de Confusion

In [ ]:
y_pred_probs = model.predict(X_val, verbose=0)
y_pred = (y_pred_probs > 0.5).astype(int).flatten()

cm = confusion_matrix(y_val, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Sano', 'Enfermo'],
            yticklabels=['Sano', 'Enfermo'])
plt.title('Matriz de Confusion - Validacion')
plt.ylabel('Real')
plt.xlabel('Predicho')
plt.tight_layout()
plt.show()

print(classification_report(y_val, y_pred, target_names=['Sano', 'Enfermo']))

### 6.3 Curva ROC

In [ ]:
fpr, tpr, _ = roc_curve(y_val, y_pred_probs)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC (AUC = {roc_auc:.3f})')
plt.plot([0,1],[0,1], color='navy', lw=2, linestyle='--')
plt.xlim([0,1])
plt.ylim([0,1.05])
plt.xlabel('Falsos Positivos')
plt.ylabel('Verdaderos Positivos (Sensibilidad)')
plt.title('Curva ROC - Validacion')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()